In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"
LANDING = f"/Volumes/{CATALOG}/bronze/landing"
SALT = "dev-only-change-me"
t = lambda n: spark.table(f"{CATALOG}.{n}")

if t("bronze.posts").filter("is_synthetic = false").count() > 0:
    dbutils.notebook.exit("Hay datos reales: sin ground truth completo, se omite la evaluación")

labels = (spark.read.json(f"{LANDING}/ground_truth/labels_batch_*.jsonl")
          .select("post_id", "true_class", "true_value_cop", F.col("venue_id").alias("true_venue_id")))
ct = (spark.read.json(f"{LANDING}/ground_truth/creators_truth_batch_*.jsonl")
      .withColumn("creator_key", F.sha2(F.concat_ws("|", F.lit(SALT), F.col("creator_id")), 256))
      .select("creator_key", "is_risky"))

c = t("silver.posts_classified").join(labels, "post_id")
m = {}

v = c.filter("venue_id IS NOT NULL AND true_venue_id IS NOT NULL")
m["venue_match_accuracy"] = v.filter("venue_id = true_venue_id").count() / v.count()

for cls in ["unclear", "disclosed_collab"]:
    sub = c.filter(F.col("pred_class") == cls)
    m[f"p_in_kind_given_{cls}"] = sub.filter("true_class = 'in_kind'").count() / sub.count()

ik = c.filter("true_class = 'in_kind' AND value_mid_cop IS NOT NULL")
m["range_coverage_in_kind"] = ik.filter("true_value_cop BETWEEN value_low_cop AND value_high_cop").count() / ik.count()

true_total = labels.filter("true_class = 'in_kind'").agg(F.sum("true_value_cop")).first()[0]
exp_total = t("gold.creator_exposure").agg(F.sum("expected_value_mid")).first()[0]
m["expected_value_error_pct"] = (exp_total - true_total) / true_total

base = t("gold.risk_cases").join(ct, "creator_key")
for col, label in [("risk_score", "score"), ("tax_exposure_mid", "exposure")]:
    for k in (10, 25, 50):
        top = base.orderBy(F.desc(col), "rank").limit(k)
        m[f"precision_at_{k}_{label}"] = top.filter("is_risky").count() / k

out = (spark.createDataFrame([(k, float(x)) for k, x in m.items()], "metric string, value double")
       .withColumn("run_ts", F.current_timestamp()))
out.write.mode("append").saveAsTable(f"{CATALOG}.gold.eval_metrics")
display(out.drop("run_ts"))